# 02 — Task 1: detailed exploratory data analysis

Explore 2017–2023 only. Notebook 05 expands to 2017–2024 after the Task 2 model/configuration freeze. Run on the authorised Education project from Jupyter. Outputs below are computed evidence, not illustrative data.

Prerequisites: notebook 00 must upload the versioned runtime manifest and source bundle under the selected run prefix (including the pinned SciencePlots wheel); notebook 01 must complete the cleaned and coverage Parquet outputs. These prerequisites are checked before analysis; this notebook has not yet been executed.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import zipfile

from IPython.display import display

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
def cloud_read(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Cloud read failed; check the authorised project, object and VM permissions.")
    return result.stdout
EXPECTED_RUN_ID = "20261012-full-01"
runtime_uri = f"gs://{BUCKET}/runs/{EXPECTED_RUN_ID}/config/runtime.json"
runtime = json.loads(cloud_read(["storage", "cat", runtime_uri]))
if runtime.get("run_id") != EXPECTED_RUN_ID:
    raise RuntimeError("Runtime manifest belongs to a different run.")
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unexpected billing account or credit.")
PROJECT_ID = runtime["project_id"]
RUN_ID = runtime["run_id"]
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
PROJECT_ROOT = Path.home() / "rainfall-project"
PROJECT_ROOT.mkdir(exist_ok=True)
bundle = PROJECT_ROOT / "source_bundle.zip"
cloud_read(["storage", "cp", runtime["source_bundle_uri"], str(bundle)])
if hashlib.sha256(bundle.read_bytes()).hexdigest() != runtime["source_bundle_sha256"]:
    raise RuntimeError("Source bundle hash differs from runtime manifest.")
with zipfile.ZipFile(bundle) as archive:
    for member in archive.namelist():
        if not (PROJECT_ROOT / member).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Unsafe archive member.")
    archive.extractall(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
from rainfall.config import load_config, config_hash
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
if config_hash(config) != runtime["config_sha256"]:
    raise RuntimeError("Experiment configuration differs from the uploaded manifest.")
billing = json.loads(cloud_read(["billing", "projects", "describe", PROJECT_ID, "--format=json"]))
from rainfall.billing import validate_runtime_credit_gate
credit_gate_status = validate_runtime_credit_gate(runtime, project_id=PROJECT_ID, billing_info=billing)
display(credit_gate_status)
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
from rainfall.runtime import spark_action, environment, require_distributed
spark = (SparkSession.builder.master("yarn").appName(f"rainfall-{RUN_ID}")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "4")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.ui.retainedJobs", "10000")
    .config("spark.ui.retainedStages", "10000")
    .config("spark.sql.ui.retainedExecutions", "10000")
    .config("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
    .getOrCreate())
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("rainfall.evidence.root", f"{ROOT}/evidence")
require_distributed(spark)
display(environment(spark))
display({"run_id": RUN_ID, "config_sha256": config_hash(config), "credit_id": runtime["credit_id"]})


In [ ]:
# Install the pinned plotting wheel into the project vendor directory if needed.
# This private, cluster-local path is visible to the active Jupyter kernel.
vendor = PROJECT_ROOT / "vendor"
scienceplots_package = vendor / "scienceplots"
if not scienceplots_package.is_dir():
    wheel = PROJECT_ROOT / "wheels" / "scienceplots-2.2.2-py3-none-any.whl"
    if not wheel.is_file():
        raise RuntimeError("The pinned SciencePlots wheel is missing from the verified project bundle.")
    vendor.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(vendor), str(wheel)], check=True)
sys.path.insert(0, str(vendor))
import scienceplots
print("SciencePlots report styles registered from the project vendor directory.")

In [ ]:
from rainfall.cleaning import coverage
from rainfall.eda import explore
from rainfall.plotting import pattern_figures, eda_figures, export_eda_tables
with spark_action(spark, RUN_ID, "task1_pretest_eda"):
    clean = spark.read.parquet(f"{ROOT}/cleaned").where("year <= 2023")
    clean = clean.persist(StorageLevel.DISK_ONLY)
    month_marker = subprocess.run(
        ["gcloud", "storage", "objects", "describe", f"{ROOT}/coverage_month/_SUCCESS"],
        capture_output=True,
        text=True,
    )
    if month_marker.returncode == 0:
        months = spark.read.parquet(f"{ROOT}/coverage_month")
        years = spark.read.parquet(f"{ROOT}/coverage_year")
        coverage_source = "cleaning notebook outputs"
    else:
        months, years, _, _ = coverage(clean)
        coverage_source = "recomputed from the committed cleaned Parquet"
    print(f"Coverage source: {coverage_source}")
    try:
        aggregates = explore(clean, months, years, config, include_test=False)
        # Materialise once. Plotting/export reads these small tables, not raw rows.
        materialised = {}
        for name, frame in aggregates.items():
            destination = f"{ROOT}/task1_pretest/{name}"
            frame.write.mode("errorifexists").json(destination)
            materialised[name] = spark.read.schema(frame.schema).json(destination)
            print(f"\n{name} — full table saved to {destination}")
            materialised[name].show(20, truncate=False)
        figure_dir = PROJECT_ROOT / "figures" / "pretest"
        pattern_figures(materialised, years.where("year <= 2023"), config, figure_dir)
        eda_figures(materialised, figure_dir)
        findings = export_eda_tables(materialised, PROJECT_ROOT / "eda_tables" / "pretest")
        display(findings)
    finally:
        clean.unpersist()
cloud_read(["storage", "rsync", "--recursive", str(PROJECT_ROOT / "eda_tables" / "pretest"), f"{ROOT}/task1_pretest/report_tables"])
cloud_read(["storage", "rsync", "--recursive", str(figure_dir), f"{ROOT}/task1_pretest/figures"])


## Read the evidence before writing conclusions

1. **Coverage:** full-calendar station coverage includes zero-observation months/years. Absent station periods may precede commissioning; this is not a measured outage rate. Compare complete station-days against both observed and calendar station-days.
2. **Distribution:** dry/wet fractions use valid records; wet quantiles and histogram condition on rain > 0 mm. These pooled summaries give more weight to stations with more valid readings. The extreme reference is the training-period wet P99, not an NEA warning threshold.
3. **Month/hour:** monthly totals are observed station totals, averaged equally over eligible stations. Hourly comparisons require both month and hour coverage ≥ the declared cutoff. Inspect station/group counts next to peaks before interpreting them.
4. **Years:** compare the same stations across all analysed years at 90%, 95% and 99% coverage. If no common cohort exists, report that result and use the descriptive view only with its changing-cohort limitation.
5. **Stations/spells:** mapped stations have fixed coordinates throughout the period; eligible years may differ. Wet spells require exact consecutive five-minute timestamps; gap/boundary-censored spells are shown separately and do not define meteorological storms.

`findings.json` records calculated peak/trough values, units and support. Exported CSVs provide the full evidence even when `.show(20)` truncates the notebook display. Each figure has PDF and PNG exports. For every job listed by `spark_action`, open the Spark Web UI / History Server, review Jobs summary and **each job detail**, and retain real screenshots with matching application/job IDs, final status, duration, stages and tasks. Saved event metadata is an index, not a screenshot.

Use the distribution, coverage and seasonal panels in the report body; detailed spatial, tail and wet-spell panels can go in the appendix. Do not treat any planned output as an observed finding.

In [ ]:
spark.stop()